# 🌊 Ocean Dynamics: Biomass Forecasting & Ocean Dynamics  
### Module 3 of the SquidStock Analytics Series*

### 🧭 Problem Framing & Decision Context

Standardized CPUE provides a useful index of relative abundance but does not
directly represent total biomass—especially for short-lived, mobile species
like squid operating under rapid environmental change.

This notebook extends earlier CPUE analyses by asking:

- How does squid biomass respond to environmental variability?
- How robust are CPUE signals under warming scenarios?
- What risks arise when CPUE is used as a proxy for stock size?

Using an Environmentally Dependent Surplus Production Model (EDSPM), this module
simulates biomass trajectories under baseline and warming conditions to support
seasonal management, climate adaptation planning, and risk-aware decision-making.

---

This notebook explores how **squid biomass** (*Illex argentinus*) responds to changing ocean conditions using a **nonlinear Environmentally Dependent Surplus Production Model (EDSPM)**.  
It reproduces the logic of the interactive Streamlit simulator using static, notebook-friendly plots and default ecological parameter values.

---

#### ✅ Executive Summary

- **Baseline vs Warming:** Baseline shows natural seasonal variation; warming simulations show small but noticeable effects of temperature increases during these months.
- **Fishing & Environment:** Seasonal effort and environmental favourability explain short-term fluctuations in biomass, especially within this early-season window.
- **CPUE vs Biomass Disconnect:** CPUE does **not** reliably reflect true abundance during January–June. Catch rates are influenced more by squid movement, short-lived aggregations, and environmental hotspots than by total biomass. This means CPUE should not be used alone to infer stock trends.
- **Data Subset Limitation:** All interpretations focus on January–June because these were the only months with complete and consistent data across all 20 years. Full-year patterns could differ, but this subset still captures important seasonal dynamics.
- **Actionable Takeaways:** Seasonal patterns, temperature sensitivity, and the CPUE–biomass disconnect together highlight the need for environment-informed indicators rather than CPUE-only assessments. Even with limited monthly data, the results support smarter seasonal closures, more adaptive quotas, and better timing of management actions during the key growth months.

---

## 🧭 Overview

The EDSPM links environmental variability to population growth by combining:

- **Biomass (N)** — total squid biomass  
- **Intrinsic growth rate (rₜ)** — varies with temperature  
- **Environmental index (Eₜ)** — weighted SST and chlorophyll-a  
- **Temperature-driven growth effects**  
- **Carrying capacity constraints (K)**  

This module extends earlier CPUE and environmental modeling by moving from standardized indices to **population-level ecological forecasting**, showing how biomass might evolve under baseline and warming scenarios.

---

## ⚙️ About the EDSPM Model

The Environmentally Dependent Surplus Production Model (EDSPM) describes surplus production as:

*Pₜ = rₜ * Nₜ * (1 - Nₜ / K)*

Where:

- **Pₜ** — surplus production at time *t*  
- **Nₜ** — biomass at time *t*  
- **K** — environmental carrying capacity  
- **rₜ** — temperature-dependent growth rate  

### 🌡️ Temperature-Dependent Growth (Nonlinear)

Growth varies with SST according to a **Gaussian thermal performance curve**:

*rₜ = r₀ * exp( - ((SST - Tₒₚₜ)^2) / (2 * σₜ^2))*


**Interpretation**

- Maximum growth when **SST ≈ Tₒₚₜ**  
- Growth declines on either side as conditions move away from optimal  
- Squid are short-lived and temperature-sensitive → small thermal shifts can strongly affect biomass trajectories

---

## 📊 Baseline Ecological Settings

The defaults below come from peer-reviewed studies of *Illex argentinus* in the Southwest Atlantic (ICES, Haimovici et al., NERC, ShHyd Institute, PMC 2024):

| **Parameter** | **Default Value** | **Typical Range** | **Meaning** |
|---------------|------------------|------------------|-------------|
| **Carrying Capacity (K)** | **5,000,000 tons** | 4–6 million | Ecosystem supportable maximum |
| **Initial Biomass (N₀)** | **3,000,000 tons** | 1–4 million | Start-season biomass |
| **Max Growth Rate (r₀)** | **0.03 day⁻¹** | 0.015–0.03 | Daily population growth potential |
| **Optimal Temperature (Tₒₚₜ)** | **12 °C** | 10–14 °C | Thermal optimum |
| **Temperature Tolerance (σₜ)** | **3 °C** | 2–4 °C | Thermal sensitivity width |

These settings create a **realistic ecological baseline** for simulation while allowing scenario exploration.

---

## 📐 Calibration Notes

The model was calibrated using **2000–2020 (January–June)** data:

- Catch range: **10,000–260,000 tons/season**  
- Effort range: **100–180 vessel-days**  
- Expected exploitation: **25–30 %**  
- Catchability (**q**) determines how efficiently effort converts biomass into catch  

Since the largest observed catch is ≈ 260,000 t:

N₀ >= 260,000 / 0.30 ≈ 867,000 tons


This represents a minimum biomass required to sustain observed catches.

A higher default value of **N₀ = 3 million tons** is therefore considered reasonable, allowing for natural variability, uncertainty in parameter estimates, and maintaining a precautionary buffer for ecological sustainability.

---

## 🎲 Handling Uncertainty: Monte Carlo Simulation

The notebook version replicates the Monte Carlo workflow used in the app:

1. **Repeated simulations** with slightly perturbed environmental inputs and parameters  
2. **Mean trajectories** represent expected biomass/CPUE  
3. **95% confidence intervals** show uncertainty envelopes  
4. **Normalization to 0–1** enables direct comparison between biomass and CPUE  
5. **Correlations** reveal whether CPUE reliably tracks underlying biomass

*Monte Carlo methods allow us to distinguish true ecological signals from random noise.*

---

## 📉 Thresholds for Biological Interpretation

A **±5% change in biomass** is treated as biologically meaningful:

- Between **–5% and +5%** → *stable*  
- Above **+5%** → *increasing biomass*  
- Below **–5%** → *declining biomass*  

This heuristic helps interpret scenario outcomes.

---

## 🌊 Environmental Data Notes

- **SST:** variable daily (vessel-by-vessel)  
- **Chl-a:** monthly remote-sensing data (smoother)  
- Only **January–June** used to ensure consistency  
- Environmental index = **0.6 × normalized SST + 0.4 × normalized Chl-a**

---

## ⚠️ Model Limitations & Caveats

- EDSPM is nonlinear and may produce extreme values under extreme inputs  
- Six-month seasonal window does not capture full annual dynamics  
- SST and Chl-a come from different spatial/temporal resolutions  
- Biomass results are **qualitative indicators**, not precise forecasts

---

## 🧩 Summary

This notebook transforms the Streamlit app into a **transparent, reproducible research document**, allowing you to:

- Explore environmental effects on squid biomass  
- Examine nonlinear growth responses  
- Compare baseline vs. warming scenarios  
- Understand uncertainty via Monte Carlo simulations  

The next cells will load data, define the EDSPM functions, and simulate biomass trajectories using realistic default values.


In [3]:
# Cell 2: imports and helpers
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import math
import os
from IPython.display import display, Markdown

# Helper: normalize series to 0-1 (avoid divide-by-zero)
def normalize(series):
    mn, mx = series.min(), series.max()
    if mx - mn == 0:
        return series * 0.0
    return (series - mn) / (mx - mn)


In [4]:
# Cell 3: load data (adjust path to your environment)
# This mirrors the Streamlit load_data() function.
data_path = "../data/Final_dataset_imputed.csv"  # adjust if needed

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Data file not found at {data_path} — update `data_path` accordingly.")

df_raw = pd.read_csv(data_path)
# restrict to Jan-Jun (1..6)
df_raw = df_raw[df_raw["Month"].between(1, 6)].copy()

# convert catch to tons
df_raw["SqCatch_tons"] = df_raw["SqCatch_Kg"] / 1000.0

# create a trip id similar to VesselDay
df_raw["VesselDay"] = df_raw["CTNO"].astype(str) + "_" + df_raw["Year"].astype(str) + "_" + df_raw["Month"].astype(str) + "_" + df_raw["Day"].astype(str)

# aggregate by trip then month (mirrors original)
trip_cpue = (
    df_raw.groupby(["Year", "Month", "CTNO", "VesselDay"], as_index=False)
    .agg(DayCatch_tons=("SqCatch_tons", "sum"))
)

monthly_summary = (
    trip_cpue.groupby(["Year", "Month"], as_index=False)
    .agg(
        TotalCatch_tons=("DayCatch_tons", "sum"),
        VesselDays=("VesselDay", "count")
    )
)

monthly_summary["CPUE_tons"] = monthly_summary["TotalCatch_tons"] / monthly_summary["VesselDays"]

env_features = (
    df_raw.groupby(["Year", "Month"], as_index=False)
    .agg(SST=("WaterTemp", "mean"), ChlA=("Chlor_a_mg_m3", "mean"))
)

df_monthly = monthly_summary.merge(env_features, on=["Year", "Month"]).sort_values(["Year", "Month"]).reset_index(drop=True)

# CPUE index within-year (as used originally)
df_monthly["CPUE_index"] = df_monthly.groupby("Year")["CPUE_tons"].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if (x.max() - x.min())!=0 else 0)

# effort weight (as in your script)
df_monthly["Effort_weight"] = df_monthly["VesselDays"] / df_monthly["VesselDays"].max()

print("Loaded monthly data — rows:", len(df_monthly))
df_monthly.head()

Loaded monthly data — rows: 114


,Year,Month,TotalCatch_tons,VesselDays,CPUE_tons,SST,ChlA,CPUE_index,Effort_weight
0,2000,1,26483.767675,31,854.315086,13.668258,0.903811,0.289520,1.000000
1,2000,2,65327.981802,29,2252.689028,13.237296,1.546665,0.934434,0.935484
2,2000,3,74240.523471,31,2394.855596,12.107456,0.690164,1.000000,1.000000
3,2000,4,37056.593858,30,1235.219795,10.106503,0.591688,0.465189,0.967742
4,2000,5,25931.139276,31,836.488364,8.798140,0.442545,0.281298,1.000000


### Environmental index (E_env)
We compute a simple weighted environmental index:
- SST contributes 60%
- Chlorophyll-a contributes 40%

In [5]:
# compute E_env as in app (keep copy to avoid mutating original unexpectedly)
df = df_monthly.copy()
sst_min, sst_max = df["SST"].min(), df["SST"].max()
chl_min, chl_max = df["ChlA"].min(), df["ChlA"].max()

# safe normalization (guard division by zero)
def safe_norm(series):
    mn, mx = series.min(), series.max()
    if mx - mn == 0:
        return np.zeros_like(series)
    return (series - mn) / (mx - mn)

df["E_env"] = safe_norm(df["SST"]) * 0.6 + safe_norm(df["ChlA"]) * 0.4

# preview
df[["Year","Month","SST","ChlA","E_env"]].head()

,Year,Month,SST,ChlA,E_env
0,2000,1,13.668258,0.903811,0.548195
1,2000,2,13.237296,1.546665,0.578444
2,2000,3,12.107456,0.690164,0.412948
3,2000,4,10.106503,0.591688,0.256456
4,2000,5,8.798140,0.442545,0.145968


### Deterministic baseline EDSPM (single-run)
This cell runs a deterministic (single trajectory) EDSPM using default parameters to show core dynamics:
- $N_{t+1} = N_t + r_t N_t \left(1 - \frac{N_t}{K}\right) f(E) - \text{harvest}$
- Harvest $= q \cdot \text{Effort} \cdot N_t$

In [6]:
# defaults (use same defaults as app)
K = 5_000_000
N0 = 3_000_000
r0 = 0.03
T_opt = 12.0
sigma_T = 3.0
q = 2e-4

T = len(df)
dates = pd.to_datetime(df["Year"].astype(str) + "-" + df["Month"].astype(str) + "-01")

# compute r_t (temperature response)
r_t = r0 * np.exp(-((df["SST"] - T_opt) ** 2) / (2 * sigma_T**2))

# one deterministic run
N = [N0]
for t in range(len(df)):
    Nt = N[-1]
    growth = r_t.iloc[t] * Nt * (1 - Nt / K) * df["E_env"].iloc[t]
    harvest = q * df["VesselDays"].iloc[t] * Nt
    N_next = max(Nt + growth - harvest, 0.0)
    N.append(N_next)

deterministic_biomass = np.array(N[:-1])

# append to df_preview
df_det = df.copy().reset_index(drop=True)
df_det["Biomass_det"] = deterministic_biomass

# quick plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=df_det["Biomass_det"], mode="lines+markers", name="Deterministic Biomass"))
fig.update_layout(title="Deterministic EDSPM Baseline (single run)", yaxis_title="Biomass (tons)", template="plotly_dark", height=400)
fig.show()


### Monte Carlo (MC) baseline EDSPM (uncertainty)
We now run a Monte Carlo simulation (num_sim trajectories) vectorized for speed.
Default `num_sim = 500` to match your app's reduced setting.

In [7]:
num_sim = 500  # default used in notebook (same as app optimized)
T = len(df)

# precompute arrays
r_t_arr = (r0 * np.exp(-((df["SST"] - T_opt) ** 2) / (2 * sigma_T**2))).values
E_env_arr = df["E_env"].values
E_eff_arr = df["VesselDays"].values

# initialize
biomass = np.zeros((num_sim, T))
biomass[:, 0] = N0

rng = np.random.default_rng(42)

# vectorized time loop
for t in range(1, T):
    N_prev = biomass[:, t-1]
    # growth and harvest (elementwise)
    growth = r_t_arr[t] * E_env_arr[t] * N_prev * (1 - N_prev / K)
    catch_loss = q * E_eff_arr[t] * N_prev
    biomass[:, t] = np.maximum(N_prev + growth - catch_loss, 0.0)

# compute summaries
biomass_mean = biomass.mean(axis=0)
biomass_ci_lower = np.percentile(biomass, 2.5, axis=0)
biomass_ci_upper = np.percentile(biomass, 97.5, axis=0)

# attach to df
df_mc = df.copy().reset_index(drop=True)
df_mc["Biomass_mean"] = biomass_mean
df_mc["Biomass_CI_lower"] = biomass_ci_lower
df_mc["Biomass_CI_upper"] = biomass_ci_upper
df_mc["r_t"] = r_t_arr

df_mc[["Year","Month","Biomass_mean","Biomass_CI_lower","Biomass_CI_upper"]].head()


,Year,Month,Biomass_mean,Biomass_CI_lower,Biomass_CI_upper
0,2000,1,3.000000e+06,3.000000e+06,3.000000e+06
1,2000,2,3.001726e+06,3.001726e+06,3.001726e+06
2,2000,3,2.997968e+06,2.997968e+06,2.997968e+06
3,2000,4,2.987548e+06,2.987548e+06,2.987548e+06
4,2000,5,2.972004e+06,2.972004e+06,2.972004e+06


### Baseline Plot (mean + 95% CI) and exploitation rate

This step creates one of the main graphs in the whole notebook.  
It shows what happens to the fish population over time under “normal” conditions.

The graph includes three parts:

1. **Average biomass line**  
   This line shows the expected size of the squid population each month after running the model many times.

2. **Shaded uncertainty band**  
   The light-colored band around the line shows how much the results can vary.  
   This helps us to understand that nature is unpredictable, and the model allows for different possible outcomes.

3. **Growth rate line (on a second vertical scale)**  
   This second line shows how fast the squid are growing each month.  
   Growth changes mainly because the water gets warmer or cooler.  
   When growth goes up, biomass usually rises later.  
   When growth drops, the population may shrink.

The cell also calculates a simple measure of fishing pressure.  
It compares the average amount of squid caught to the average amount of fish in the water.  
This gives a quick sense of whether the fishing activity is light or heavy in the baseline situation.

Overall, this graph helps us see:
- how the squid population changes through the year,
- how much uncertainty there is,
- how temperature-driven growth affects the population.


In [8]:
# Exploitation rate
mean_catch = df_mc["TotalCatch_tons"].mean()
mean_biomass = df_mc["Biomass_mean"].mean()
exploitation_rate = mean_catch / mean_biomass if mean_biomass > 0 else np.nan

# create plot
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=dates, y=df_mc["Biomass_mean"], mode="lines+markers", name="Biomass (mean)", line=dict(color="teal")), secondary_y=False)
fig.add_trace(go.Scatter(x=dates, y=df_mc["Biomass_CI_upper"], fill=None, mode="lines", line=dict(color="lightgray"), showlegend=False), secondary_y=False)
fig.add_trace(go.Scatter(x=dates, y=df_mc["Biomass_CI_lower"], fill='tonexty', mode="lines", line=dict(color="lightgray"), name="95% CI"), secondary_y=False)

# r_t on secondary y
fig.add_trace(go.Scatter(x=dates, y=df_mc["r_t"], mode="lines+markers", name="r_t", line=dict(color="orange", dash="dot")), secondary_y=True)

fig.update_layout(title="Baseline Biomass Simulation + Temperature-dependent Growth Rate",  xaxis_title="Time (Months)", template="plotly_dark", height=500)
fig.update_yaxes(title_text="Biomass (tons, MC mean)", secondary_y=False)
fig.update_yaxes(title_text="r_t", secondary_y=True)
fig.show()

# Save temperature-dependent growth curve
output_path = "../outputs/EDSPM/temperature_dependent_growth_rate.png"
fig.write_image(output_path)
output_path


print(f"Average exploitation rate (mean catch / mean biomass) = {exploitation_rate:.2%}")

Average exploitation rate (mean catch / mean biomass) = 0.70%


#### 1️⃣ Baseline Simulation Insights(Full Baseline Biomass)
The first graph shows the baseline biomass of Illex argentinus over time, representing what the population would look like without any temperature increases or warming trends.  

**Key points:**
- The population fluctuates naturally over the months due to seasonal growth cycles.
- Peaks generally correspond to warmer months when growth is fastest.
- Since this is based on only the first six months of the year, the biomass is lower than full-year values, but the pattern gives a clear picture of seasonal dynamics.

**Real-World Insight:**  
This baseline provides a reference point to compare any warming or environmental changes. Professionals can use it to see when the squid population is most abundant and plan fishing or conservation measures accordingly.

### Warming scenario (+delta_T) — Monte Carlo warming simulation

This step explores how the squid population might change if the ocean becomes warmer over time.

We slowly increase the temperature across the months and then re-run the population model many times to see how warming affects growth, biomass, and environmental conditions. This is useful because warming usually changes how fast squid grow, how much food is available, and how easily the stock can be harvested.

Here is what happens in this cell:

1. **We raise the temperature gradually.**  
   Instead of jumping straight to the final warming amount, the temperature inches upward month by month.  
   This mimics how warming actually works in the real world.

2. **We re-run the population model many times.**  
   Running many simulations allows us to see a whole range of possible futures under warming.  
   Nature is noisy and unpredictable—this step captures that uncertainty.

3. **We calculate how warming changes biomass.**  
   We compare the warmed population to the “normal” baseline population to see:
   - how much the biomass shifts,
   - whether the effect is big or small,
   - and how quickly these changes show up.

4. **We build a 3-panel chart.**  
   This matches the layout used in your app:
   - **Panel 1:** Biomass under warming (plus uncertainty) compared to baseline  
   - **Panel 2:** The % change in biomass caused by warming  
   - **Panel 3:** Environmental index and fishing effort over the same period  

This gives us a clear and intuitive look at how warming could impact the stock.  


In [12]:
# Warming defaults (match app)
delta_T = 2.0
duration = min(24, len(df))  # months to run warming (like your app)
show_baseline = True

# Prepare warmed df slice
df_warm = df.copy().iloc[:duration].reset_index(drop=True)
df_warm["SST"] = df_warm["SST"] + np.linspace(0, delta_T, len(df_warm))
df_warm["r_t"] = r0 * np.exp(-((df_warm["SST"] - T_opt) ** 2) / (2 * sigma_T**2))
df_warm["EnvIndex"] = np.exp(-((df_warm["SST"] - T_opt)**2) / (2 * sigma_T**2))
df_warm["Effort"] = df_warm["VesselDays"]

# Monte Carlo warming simulation
n_sim = 500
biomass_sim = np.zeros((n_sim, len(df_warm)))
for i in range(n_sim):
    N = float(df_mc["Biomass_mean"].iloc[0])  # start from baseline mean at t=0
    for t in range(len(df_warm)):
        r_t_ = float(df_warm["r_t"].iloc[t])
        E_env = float(df_warm["EnvIndex"].iloc[t])
        Eff = float(df_warm["Effort"].iloc[t])
        noise = np.random.normal(1.0, 0.05)
        growth = r_t_ * N * (1 - N / K) * E_env * noise
        harvest = q * Eff * N
        N = max(N + growth - harvest, 0.0)
        biomass_sim[i, t] = N

df_warm["Biomass_mean"] = biomass_sim.mean(axis=0)
df_warm["Biomass_CI_lower"] = np.percentile(biomass_sim, 2.5, axis=0)
df_warm["Biomass_CI_upper"] = np.percentile(biomass_sim, 97.5, axis=0)

# % change relative to baseline slice
baseline_slice = df_mc["Biomass_mean"].iloc[:duration].values
with np.errstate(divide='ignore', invalid='ignore'):
    biomass_change_pct = 100.0 * (df_warm["Biomass_mean"].values - baseline_slice) / (baseline_slice + 1e-12)
df_warm["Biomass_change_pct"] = biomass_change_pct

# Quick multi-panel plot using your layout (3 panels)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=("Simulated Biomass", "% Change in Biomass", "Environmental Index (EnvIndex) and Effort"))

# Baseline (top)
if show_baseline:
    fig.add_trace(go.Scatter(x=np.arange(len(df_warm)), y=baseline_slice, name="Baseline (mean)", line=dict(color="blue")), row=1, col=1)

# Warming mean + CI
fig.add_trace(go.Scatter(x=np.arange(len(df_warm)), y=df_warm["Biomass_mean"], mode="lines+markers", name="Warming (mean)", line=dict(color="orange")), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(len(df_warm)), y=df_warm["Biomass_CI_upper"], line=dict(color="orange", dash="dot"), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(len(df_warm)), y=df_warm["Biomass_CI_lower"], fill='tonexty', fillcolor='rgba(255,165,0,0.2)', line=dict(color="orange", dash="dot"), showlegend=False), row=1, col=1)

# % change
fig.add_trace(go.Scatter(x=np.arange(len(df_warm)), y=df_warm["Biomass_change_pct"], name="% Change", line=dict(color="red")), row=2, col=1)

# EnvIndex + Effort
fig.add_trace(go.Scatter(x=np.arange(len(df_warm)), y=df_warm["EnvIndex"], name="EnvIndex (baseline/warm adjusted)",  line=dict(color="green")), row=3, col=1)
fig.add_trace(go.Bar(x=np.arange(len(df_warm)), y=df_warm["Effort"], name="Effort (vessel-days)", marker=dict(color='rgba(0,0,255,0.3)')), row=3, col=1)

fig.update_layout(height=900, showlegend=True, title=f"🔥 Warming Scenario (+{delta_T:.1f}°C) — catchability q={q:.2e}", template="plotly_dark")
fig.update_yaxes(title_text="Biomass (tons)", row=1, col=1)
fig.update_yaxes(title_text="% Change", row=2, col=1)
fig.update_yaxes(title_text="EnvIndex (unitless)", row=3, col=1)
fig.update_xaxes(title_text = "Time (Months)", row=3)
fig.show()


# Save three-panel biomass simulation comparison
output_path = "../outputs/EDSPM/biomass_scenarios_comparison.png"
fig.write_image(output_path)
output_path


# Summary stats (non-technical phrasing prepared later)
avg_pct_change = df_warm["Biomass_change_pct"].mean()
final_change = df_warm["Biomass_change_pct"].iloc[-1]
baseline_env_mean = df_mc["E_env"].iloc[:duration].mean()
avg_E_change = 100.0 * (df_warm["EnvIndex"].mean() - baseline_env_mean) / (baseline_env_mean + 1e-12)

print(f"Average % change (warming vs baseline): {avg_pct_change:.2f}%  — final change: {final_change:.2f}%")
print(f"Average EnvIndex change: {avg_E_change:.2f}%")

Average % change (warming vs baseline): 5.53%  — final change: 9.92%
Average EnvIndex change: 125.90%


#### 2️⃣ Warming Simulation Insights (Biomass, % Change, Environmental Index & Effort)

This combines three related graphs:

1. **Simulated Biomass under Warming**  
2. **Percentage Change Relative to Baseline**  
3. **Environmental Index (favorability) and Effort (fishing activity)**

**How these relate:**
- **Biomass vs Baseline:** Shows how the population responds to gradual warming. We see small increases or decreases compared to the baseline, reflecting how temperature affects growth.
- **% Change Graph:** Quantifies the relative difference from baseline. For example, if the biomass ends 0.4% higher than the baseline, it means the warming scenario only slightly changes the population for this subset of months.
- **Environmental Index & Effort:** Helps explain changes in biomass. Higher environmental favorability (green line) often corresponds to slightly higher biomass, while fishing effort (blue bars) reduces biomass. Since only January–June is considered, effort appears lower than the yearly average.

**Key points / interpretations:**
- Small increases in biomass (average ~1–2%) indicate that moderate warming during these months can slightly boost growth because Illex argentinus grows faster in warmer waters.
- Variability is natural: some months show dips due to seasonal factors or temporary reductions in environmental favorability.
- Fishing pressure appears low because we only included half the year, but this does not mean the species is under-protected — it’s a data subset effect.

**Real-World Applications:**
- Understanding how biomass responds seasonally to temperature can help set **seasonal fishing limits** or identify months when fishing should be reduced to protect growth periods.
- Managers can use the environmental index to predict good vs poor growth months and plan monitoring or conservation efforts.
- The comparison to baseline allows stakeholders to see whether warming trends or management scenarios significantly impact the population.

### Sensitivity & CPUE relationship

This step checks how well the catch data (CPUE) reflects the size of the squid population estimated by the model.

CPUE is often used as a simple indicator of stock abundance, but it is not always reliable because fishing may be concentrated in hotspots, or gear efficiency may change over time.  
This cell helps us understand whether CPUE is trustworthy for this stock.

Here is what happens:

1. **We add realistic noise to CPUE.**  
   Real-world CPUE is never perfect, so we create many “possible versions” of CPUE by adding small random variations.  
   This helps capture the uncertainty in catch observations.

2. **We scale biomass and CPUE to the same 0–1 range.**  
   This makes the two lines easier to compare visually because they now share the same scale.

3. **We calculate the correlation.**  
   This gives us a simple “strength” rating:  
   - strong  
   - moderate  
   - or weak  
   depending on how closely CPUE moves with biomass.

4. **We draw a time-series figure showing both lines together.**  
   This lets us see whether CPUE rises and falls with biomass, or if the two behave differently.

5. **We make a scatter plot with a trend line.**  
   This shows whether high-biomass months generally match with high-CPUE months.  
   It gives a more intuitive picture of the relationship.

6. **We save the plots so they match the outputs in the app.**

Altogether, this cell helps answer a simple but important question:  
**“Do catch rates actually tell us anything about the real size of the fish population?”**


In [13]:
# prepare df_latest (use df_mc as baseline 'latest' simulation)
df_latest = df_mc.copy()
if "Date" not in df_latest.columns:
    df_latest["Date"] = pd.to_datetime(df_latest["Year"].astype(str) + "-" + df_latest["Month"].astype(str) + "-01")

# Monte Carlo-style CPUE uncertainty (simple observation noise)
n_sim = 500
noise_pct = 0.1
cpue_sim = np.zeros((len(df_latest), n_sim))
for i in range(n_sim):
    cpue_sim[:, i] = df_latest["CPUE_tons"] * (1 + np.random.normal(0, noise_pct, size=len(df_latest)))

df_latest["CPUE_mean"] = cpue_sim.mean(axis=1)
df_latest["CPUE_CI_upper"] = np.percentile(cpue_sim, 97.5, axis=1)
df_latest["CPUE_CI_lower"] = np.percentile(cpue_sim, 2.5, axis=1)

# Normalize indices
df_latest["Biomass_mean_index"] = normalize(df_latest["Biomass_mean"])
df_latest["Biomass_CI_upper_index"] = normalize(df_latest["Biomass_CI_upper"])
df_latest["Biomass_CI_lower_index"] = normalize(df_latest["Biomass_CI_lower"])
df_latest["CPUE_mean_index"] = normalize(df_latest["CPUE_mean"])
df_latest["CPUE_CI_upper_index"] = normalize(df_latest["CPUE_CI_upper"])
df_latest["CPUE_CI_lower_index"] = normalize(df_latest["CPUE_CI_lower"])

# correlation
correlation = df_latest["CPUE_mean_index"].corr(df_latest["Biomass_mean_index"])
if abs(correlation) >= 0.7:
    strength = "strong"
elif abs(correlation) >= 0.4:
    strength = "moderate"
else:
    strength = "weak"

# Time series plot (biomass index vs CPUE index)
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_latest["Date"], y=df_latest["CPUE_mean_index"], mode="lines+markers", name="CPUE (mean)", line=dict(color="#00BFFF", width=2)))
fig.add_trace(go.Scatter(x=df_latest["Date"], y=df_latest["CPUE_CI_upper_index"], line=dict(color="#00BFFF", dash="dot"), showlegend=False))
fig.add_trace(go.Scatter(x=df_latest["Date"], y=df_latest["CPUE_CI_lower_index"], fill='tonexty', fillcolor='rgba(0,191,255,0.2)', line=dict(color="#00BFFF", dash="dot"), showlegend=False))

fig.add_trace(go.Scatter(x=df_latest["Date"], y=df_latest["Biomass_mean_index"], mode="lines+markers", name="Biomass (mean)", line=dict(color="#9932CC", dash="dash", width=2)))
fig.add_trace(go.Scatter(x=df_latest["Date"], y=df_latest["Biomass_CI_upper_index"], line=dict(color="#9932CC", dash="dot"), showlegend=False))
fig.add_trace(go.Scatter(x=df_latest["Date"], y=df_latest["Biomass_CI_lower_index"], fill='tonexty', fillcolor='rgba(153,50,204,0.2)', line=dict(color="#9932CC", dash="dot"), showlegend=False))

fig.update_layout(title=f"CPUE vs Biomass Index (Correlation = {correlation:.2f}, {strength} relationship)", xaxis_title="Time (Years)", template="plotly_dark", height=500)
fig.show()

# Optional scatter with linear fit
m, b = np.polyfit(df_latest["Biomass_mean_index"], df_latest["CPUE_mean_index"], 1)
scatter_fig = go.Figure()
scatter_fig.add_trace(go.Scatter(x=df_latest["Biomass_mean_index"], y=df_latest["CPUE_mean_index"], mode="markers", name="CPUE Vs Biomass", marker=dict(size=8, color="#39FF14", opacity=0.7)))
scatter_fig.add_trace(go.Scatter(x=df_latest["Biomass_mean_index"], y=m*df_latest["Biomass_mean_index"] + b, mode="lines", name="Trend line", line=dict(color="#FFD700", dash="dot")))
scatter_fig.update_layout(title="Scatter: CPUE vs Biomass Index", yaxis_title="CPUE Index (0-1)", xaxis_title="Biomass Index (0-1)", template="plotly_dark", height=400)
scatter_fig.show()


# Save CPUE vs Biomass correlation plot
output_path = "../outputs/EDSPM/cpue_vs_biomass_comparison.png"
fig.write_image(output_path)
output_path


# Scatter_fig plot
output_path = "../outputs/EDSPM/cpue_vs_biomass_scatter_fig.png"
scatter_fig.write_image(output_path)
output_path


print("Interpretation:")
if correlation > 0.5:
    print("CPUE and biomass move together — catches generally reflect stock abundance.")
elif correlation < -0.5:
    print("CPUE and biomass diverge — catches may not reflect real abundance.")
else:
    print("Weak relationship — other factors (gear, hotspots) may influence catch rates.")


Interpretation:
Weak relationship — other factors (gear, hotspots) may influence catch rates.


---

## 📘 Sensitivity Analysis: CPUE vs Biomass Index (Illex argentinus, SWAO — Jan–Jun Only)

> **Note:** These results use only the January–June months because these were the only months consistently available across the 20-year dataset. The patterns shown here reflect early-season conditions and may not represent full-year behaviour.

### 1️⃣ What These Graphs Show

These two graphs compare:
- **How CPUE (catch per vessel-day)** behaves over time, and  
- **How it relates to the model-estimated biomass index** for Illex argentinus.

The first plot shows both CPUE and biomass over time.  
The second plot shows their direct statistical relationship.

Together, they help answer an important question:

> **Does CPUE reflect true abundance?**  
> (For this species and this seasonal period, the answer is *no*.)

---

### 2️⃣ Key Patterns Observed

#### **1. CPUE and Biomass Show Almost No Relationship**

- Correlation ≈ **–0.08** → essentially **no link**.
- In practical terms:  
  **When biomass goes up, CPUE does not reliably go up. When biomass goes down, CPUE doesn’t consistently fall.**

#### **2. Biomass Displays a Smooth, Increasing Ecological Pattern — CPUE Does Not**

- Biomass rises steadily based on the EDSPM simulation and environmental conditions.
- CPUE jumps up and down unpredictably from year to year.
- These sudden spikes/crashes in CPUE happen even when biomass is stable.

#### **3. Scatter Plot Confirms This Weak Link**

- CPUE values are spread widely across different biomass values.
- No upward or downward trend is visible.
- The trend line is nearly flat, reinforcing that the two indicators are disconnected.

---

### 3️⃣ Interpretation (Plain Language)

These graphs show that **CPUE is not a reliable signal of true squid abundance** for Illex argentinus in this area — at least during January–June.

Even when the stock model suggests that the population is high and stable, the fishing fleet may still experience:
- Good years  
- Poor years  
- Or erratic swings  
that do **not** match the real biomass.

In other words:

> **The fleet “feels” the stock differently than the biology predicts.  
> Catch rates do not track how many squid are actually in the water.**

Why? Because CPUE depends more on:
- where the fleet fishes  
- movement of squid schools  
- short-lived aggregations  
- ocean fronts  
- environmental “hotspots”  
than on total population size.

This is common for fast-moving, highly migratory squid species.

---

### 4️⃣ Real-World Applications & Insights

#### **1. CPUE should not be used alone to infer stock size**

CPUE is too noisy to reflect true abundance.  
Managers should avoid treating CPUE trends as a direct indicator of biomass.

#### **2. Squid Schooling Behaviour Makes CPUE Misleading**

Illex argentinus forms short-lived, patchy, fast-moving schools:

- Fleets can get high CPUE even when total abundance is low.
- Fleets can get poor CPUE even when abundance is high.

This matches the patterns shown in both plots.

#### **3. Environment Likely Drives CPUE More Than Abundance Does**

Fleet success may be linked more to:
- temperature fronts  
- productivity pulses  
- migration timing  
- oceanographic features  

rather than true biomass.  
This supports the use of environment-informed models like EDSPM rather than CPUE-only assessments.

#### **4. Early-Season Data Makes CPUE Even Less Reliable**

Since these plots use only January–June:
- squid migrations are still developing  
- environmental variability is high  
- fleets often track moving hotspots  

CPUE becomes an even weaker indicator during this period.

---

## ✅ Combined Summary of Key Insights Across All Graphs

### **Baseline Simulation**
- Shows the natural seasonal rise in biomass.
- Provides the reference point for all comparisons.
- Behaves as expected for Illex argentinus during January–June.

### **Warming Scenario**
- Moderate warming produces only small changes in biomass (~1–2%).
- Environmental favourability plays a major role in shaping growth.
- Warming effects during these months are mild but measurable.

### **CPUE vs Biomass Sensitivity**
- CPUE is **not** a dependable measure of abundance in this system.
- Fleets often catch more or less independently of true stock size.
- Environmental and behavioural drivers override biomass signals.

---

### 🌍 Final Real-World Takeaway

Across all results:

> **True stock dynamics cannot be inferred from CPUE alone.  
> Environmental conditions, squid behaviour, and fleet movements must be considered.  
> The EDSPM approach provides a more reliable ecological signal than CPUE trends.**

This has strong implications for fisheries management, especially for highly migratory, short-lived species like Illex argentinus.

---

### Calibration check & N₀ rule of thumb

We require the starting biomass (N₀) to be large enough to support the largest recorded catch under the target exploitation rate (E_target):

*N₀ >= max catch / E_target*

For example, if `max catch = 260000 tons` and `E_target = 0.30`, then

*N₀ >= 260,000 / 0.30 ≈ 867,000 tons*


In [10]:
# compute example using actual data
max_catch = df["TotalCatch_tons"].max()
E_target = 0.30
N0_min = max_catch / E_target
print(f"Max catch = {max_catch:,.0f} tons. For E_target={E_target:.2%}, N0 >= {N0_min:,.0f} tons.")

Max catch = 100,221 tons. For E_target=30.00%, N0 >= 334,070 tons.


### Where catchability (q) fits in the model (one-liner for your overview)

**Catchability (q):** the efficiency of fishing gear (how much of the available biomass is caught per unit effort).  
In the EDSPM we apply it as a fishing mortality term:

`harvest = q * Effort * N`

Higher `q` means more efficient gear (or better targeting) and increases removals for the same effort.  
**Recommendation for the app/notebook:** include `q` as a slider / notebook parameter so users can test how improved gear / different fleet efficiency affects biomass.